# 29 · Observability: traces, what to log, and a dashboard
Level L5 · Part 5 · core · ~20 min · builds on 06

## Goal
Trace every model call and every tool call of the support agent as **spans** in one **trace** per task. Log only what
is safe to keep. Turn the spans into a dashboard row for each task: did it answer, what did it cost, how long did it take.

## Why it matters
A customer writes: "The agent gave me the wrong refund answer yesterday." What do you have? Without traces, you have
the final text and nothing else. You cannot tell which step ran, which tool failed, how many model calls it took or
what it cost. Your only option is to read the customer's chat and guess.

| Question about one task | Without traces | With traces |
|---|---|---|
| Which tool did it call, and did it fail? | Guess from the answer | The `execute_tool` span, status ERROR |
| How many model calls, and what did they cost? | Your bill for the day | The `chat` spans, token counts on each |
| Where did the 40 seconds go? | Nowhere | The span durations, parent to child |

**What it makes easy:** one tree per task that you can open in seconds, and totals you can add up across tasks.
**When you don't need it:** a single local test you run and read yourself. Once real customers call the agent, you do.

## The idea
A **span** is one timed step with a name and attributes, such as one model call. A **trace** is the tree of spans for
one task: the agent span at the top, with its model and tool spans under it. **OpenTelemetry** is the open standard
for spans. Its **GenAI conventions** name the attributes of model and tool steps, such as the model name and the token
counts. Those names are still marked "Development", so pin the library version.

Logging is a separate choice. Keep counts, ids, hashes, model names and status. Do not keep the customer's words.
A dashboard then reads the spans: success rate, cost per task and p95 latency (the time that 95 out of 100 tasks beat).

Analogy: a flight recorder for each trip. Every leg is timed and the tape records the engines, not the passengers' names.

## Picture
```mermaid
flowchart TD
  A["invoke_agent support-desk (one task)"] --> C1["chat: model asks for two lookups"]
  A --> T1["execute_tool lookup_order"]
  A --> T2["execute_tool lookup_order"]
  A --> C2["chat: model writes the answer"]
```

## Step by step
**Step 1 · the tracer (no call).** The tracer is the OpenTelemetry SDK. `ConsoleSpanExporter` prints spans by default;
its `out=` argument sends them to a file instead (opened with "w", so each run starts empty), and its formatter writes
each span as one JSON line in `data/29/traces.jsonl`.
`SimpleSpanProcessor` hands each span to the exporter as soon as the span ends. The dashboard below reads that file.
The model and its prices are the ones from lesson 06.

In [1]:
import sys; sys.path.insert(0, "..")
import hashlib, json, math
from datetime import datetime
from pathlib import Path

from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor
from opentelemetry.trace import SpanKind, Status, StatusCode

MODEL = "gemini-3.1-flash-lite"    # the course's default model, the same as lessons 02 to 06
PRICE_IN, PRICE_OUT = 0.25, 1.50   # USD per 1M tokens, as checked for lesson 02 on 2026-10-20
TRACE_FILE = Path("../data/29/traces.jsonl")  # written by this lesson's run
TRACE_FILE.parent.mkdir(parents=True, exist_ok=True)

provider = TracerProvider()  # in production, add an OTLP exporter here instead
exporter = ConsoleSpanExporter(out=TRACE_FILE.open("w"), formatter=lambda span: span.to_json(indent=None) + "\n")
provider.add_span_processor(SimpleSpanProcessor(exporter))  # export each span as soon as it ends
tracer = provider.get_tracer("brightside-support-desk")
print("tracer ready; spans go to", TRACE_FILE.name)

tracer ready; spans go to traces.jsonl


**Step 2 · the tool (no call).** The tool is `lookup_order` from lesson 06, over the same fictional rows. `run_tool`
runs each tool request inside its own `execute_tool` span. The arguments are not logged. A tool error is still
returned to the model as text, and the span gets the status ERROR.

In [2]:
ORDERS = {  # the same fictional rows as lessons 05 and 06
    "1093": {"status": "in transit", "days_late": 3},
    "2207": {"status": "delivered", "days_late": 0},
}


def run_tool(name: str, args: dict) -> dict:  # this lesson has one tool, lookup_order
    with tracer.start_as_current_span(f"execute_tool {name}", kind=SpanKind.INTERNAL) as span:
        span.set_attribute("gen_ai.operation.name", "execute_tool")
        span.set_attribute("gen_ai.tool.name", name)
        result = ORDERS.get(args.get("order_id", ""), {"error": "no order with that id"})
        if "error" in result:  # the model reads this text; the span is marked failed
            span.set_status(Status(StatusCode.ERROR, "tool returned an error"))
        return result

**Step 3 · the model client (setup).** The client is the one from lesson 06: the same tool declaration, and
`llm_replay` so that every call is recorded once and replayed without a key. The key is read from `.env`, never
typed here.

In [3]:
import llm_replay  # course helper: records and replays model calls (not part of the SDK)
llm_replay.start("29_observability")

from google import genai  # the official Google SDK
from google.genai import types
from tut_env import env  # course helper: reads a setting from the .env file (not part of the SDK)

client = genai.Client(vertexai=True, api_key="replay" if llm_replay.is_replaying() else env("GOOGLE_CLOUD_API_KEY"))
TOOL_CONFIG = types.GenerateContentConfig(temperature=0, tools=[types.Tool(function_declarations=[
    types.FunctionDeclaration(name="lookup_order", description="Look up the status of one order by its number.",
                              parameters_json_schema={"type": "object", "properties": {"order_id": {"type": "string"}},
                                                      "required": ["order_id"]})])])

LLM mode: replay (29_observability.yaml)


**Step 4 · one traced task.** `ask` runs the loop from lesson 06 in a small form, with a step cap of 4. Each model
call gets a `chat` span with the model and its token counts. Output tokens are the answer tokens plus the hidden
thinking tokens, and both are billed as output. Each tool request gets an `execute_tool` span from `run_tool`. The
whole task is one `invoke_agent` span. The question is stored as a hash, and the customer's text is not written to any span.

In [4]:
def chat_step(history: list):  # one model call = one chat span; returns the reply, its turn and its tool requests
    with tracer.start_as_current_span(f"chat {MODEL}") as step:
        resp = client.models.generate_content(model=MODEL, contents=history, config=TOOL_CONFIG)
        u = resp.usage_metadata  # the token counts the reply already carries
        step.set_attributes({"gen_ai.operation.name": "chat", "gen_ai.request.model": MODEL,
                             "gen_ai.usage.input_tokens": u.prompt_token_count or 0,
                             "gen_ai.usage.output_tokens": (u.candidates_token_count or 0) + (u.thoughts_token_count or 0)})
    turn = resp.candidates[0].content
    return resp, turn, [p.function_call for p in turn.parts if p.function_call]

The loop itself: each pass makes one `chat` span, then runs the tools it asked for, each in its own span.

In [5]:
def ask(question: str, task_id: str, max_steps: int = 4) -> str:
    with tracer.start_as_current_span("invoke_agent support-desk") as agent:
        agent.set_attributes({"gen_ai.operation.name": "invoke_agent", "gen_ai.conversation.id": task_id,
                              "question.sha256": hashlib.sha256(question.encode()).hexdigest()[:16]})
        history = [types.Content(role="user", parts=[types.Part(text=question)])]
        for _ in range(max_steps):
            resp, turn, calls = chat_step(history)
            if not calls:  # no tool request: the model has answered
                agent.set_status(Status(StatusCode.OK))
                return resp.text
            history.append(turn)
            results = [types.Part.from_function_response(name=c.name, response=run_tool(c.name, dict(c.args)))
                       for c in calls]
            history.append(types.Content(role="user", parts=results))
        agent.set_status(Status(StatusCode.ERROR, "step cap reached"))
        return "stopped: step cap reached"

**Step 5 · two tasks (2 model calls each).** Task 1 asks about two orders the tool knows. Task 2 asks about an order
that does not exist. The tool returns an error, the model reads it and answers, and the trace marks the lookup as
failed while the task still answers. That difference matters for the dashboard. The two tasks make 4 model calls in
total, and `llm_replay.stop()` runs at the end of this cell.

In [6]:
QUESTIONS = {
    "task-1": "Where are my orders 1093 and 2207? Check both, then answer in two sentences.",
    "task-2": "Where is order 9999? Check it and tell me what you found.",
}
for task_id, question in QUESTIONS.items():
    print(task_id, "->", ask(question, task_id)[:110])
llm_replay.stop()  # the last model call of this notebook

task-1 -> Order 1093 is currently in transit and is three days late. Order 2207 has been successfully delivered.
task-2 -> I'm sorry, but I couldn't find an order with the ID 9999. Please double-check the order number and try again.


**Step 6 · read the trace (no call).** `load_spans` reads the spans of this run. If this run made no model calls (no
key, no recording), it reads the spans saved from the recorded run instead. `walk` prints every trace as a tree:
the parent and child structure, each step's name and status. A status of UNSET means nothing marked the span as
failed; only ERROR is a failure.

In [7]:
def load_spans() -> list[dict]:
    if TRACE_FILE.read_text().strip():  # this run made model calls
        return [json.loads(line) for line in TRACE_FILE.read_text().splitlines()]
    print("no model calls this run: using the recorded output")
    return json.loads(Path("../data/recorded/29.json").read_text())  # recorded output, saved at build time


spans = load_spans()


def walk(parent_id, depth: int = 0) -> None:
    for s in [s for s in spans if s["parent_id"] == parent_id]:
        print("  " * depth + f"{s['name']}  {s['status']['status_code']}")
        walk(s["context"]["span_id"], depth + 1)


walk(None)

invoke_agent support-desk  OK
  chat gemini-3.1-flash-lite  UNSET
  execute_tool lookup_order  UNSET
  execute_tool lookup_order  UNSET
  chat gemini-3.1-flash-lite  UNSET
invoke_agent support-desk  OK
  chat gemini-3.1-flash-lite  UNSET
  execute_tool lookup_order  ERROR
  chat gemini-3.1-flash-lite  UNSET


**Step 7 · what to log (no call).** A log line keeps a task id, a short hash of the question and its length. The hash
lets you match two logs of the same question. It does not hide a short or guessable question, such as an order
number, so treat it as a link, not as a secret. The last check reads every real span and counts any `@` sign, which
would mean an email address got in.

In [8]:
RAW_TICKET = "Refund my order 2207, my email is ruth@example.com and I want it today."
safe_fields = {"task": "task-3", "question_sha256": hashlib.sha256(RAW_TICKET.encode()).hexdigest()[:16],
               "question_chars": len(RAW_TICKET)}
print("unsafe log line:", {"task": "task-3", "question": RAW_TICKET})
print("safe log line:  ", safe_fields)
print("spans in this trace file that contain an @ sign:", sum("@" in json.dumps(s) for s in spans))

unsafe log line: {'task': 'task-3', 'question': 'Refund my order 2207, my email is ruth@example.com and I want it today.'}
safe log line:   {'task': 'task-3', 'question_sha256': 'bb841c1a6a1a827c', 'question_chars': 71}
spans in this trace file that contain an @ sign: 0


**Step 8 · the dashboard (no call).** One row per task, built from its spans: did it answer (the agent span is OK),
how many model calls and tool errors it had, its cost from the token counts on the `chat` spans, and its latency from
the agent span. The summary gives the success rate, the cost per task, the p95 latency and an SLO check. An SLO is a
target you commit to, here 95% of tasks answered. Replayed calls return at once, so the latencies below are not real
model latency, and they change each run. Over two tasks, the p95 is just the slower task. Read the numbers as the
shape of the dashboard, not as a measurement.

In [9]:
SLO_SUCCESS = 0.95


def ms(span: dict) -> float:
    return (datetime.fromisoformat(span["end_time"]) - datetime.fromisoformat(span["start_time"])).total_seconds() * 1000


def p95(values: list[float]) -> float:  # nearest rank: the value 95% of tasks are at or below
    return sorted(values)[math.ceil(0.95 * len(values)) - 1]


def task_rows(spans: list[dict]) -> list[dict]:
    rows = []
    for root in [s for s in spans if s["parent_id"] is None]:  # one invoke_agent span per task
        mine = [s for s in spans if s["context"]["trace_id"] == root["context"]["trace_id"]]
        chats = [s["attributes"] for s in mine if s["attributes"].get("gen_ai.operation.name") == "chat"]
        tools = [s for s in mine if s["attributes"].get("gen_ai.operation.name") == "execute_tool"]
        cost = sum(a["gen_ai.usage.input_tokens"] * PRICE_IN + a["gen_ai.usage.output_tokens"] * PRICE_OUT
                   for a in chats) / 1_000_000
        rows.append({"task": root["attributes"]["gen_ai.conversation.id"],
                     "answered": root["status"]["status_code"] == "OK", "model_calls": len(chats),
                     "tool_errors": sum(t["status"]["status_code"] == "ERROR" for t in tools),
                     "cost_usd": cost, "latency_ms": ms(root)})
    return rows

The summary of the dashboard rows:

In [10]:
rows = task_rows(spans)
success = sum(r["answered"] for r in rows) / len(rows)
print(f"success rate {success:.0%} | cost per task ${sum(r['cost_usd'] for r in rows) / len(rows):.6f} "
      f"| p95 latency {p95([r['latency_ms'] for r in rows]):.0f} ms")
print("SLO breached (alert)" if success < SLO_SUCCESS else "within SLO")

success rate 100% | cost per task $0.000130 | p95 latency 54 ms
within SLO


## At work
- **Teams open the trace before the prompt.** A complaint arrives, and on-call opens the trace of that task to see the
  failing step. A hand-written loop emits no tool spans until you add them, as `run_tool` does here. → lesson 37
- **Logs hold hashes and counts, not ticket text.** A ticket holds names and emails, and many people read trace stores.
  A poisoned ticket can also push the agent into a tool call it should not make; the trace shows that call, and
  policy checks and an audit log stop it. → lesson 31
- **Cost per task and p95 latency are the numbers finance and on-call watch.** Cost comes from the token counts on each
  `chat` span; cache reads are priced differently, so count them separately. → lesson 33
- **Trace UIs and vendor SDKs.** Any OTLP-compatible trace UI can read these spans, such as Phoenix or Langfuse.
  Go deeper: a self-hosted trace UI and trace sampling.

## Common mistakes
**Mistake: reading a span's duration before the span ends.** The end time is not set until the span ends, so the
subtraction fails. This cell uses its own provider, so no probe span reaches the trace file.

In [11]:
probe = TracerProvider().get_tracer("probe").start_span("chat probe")
print("duration ms:", (probe.end_time - probe.start_time) / 1e6)  # the span has not ended yet

TypeError: unsupported operand type(s) for -: 'NoneType' and 'int'

The fix is to call `end()` first. Then the duration is a number.

In [12]:
probe = TracerProvider().get_tracer("probe").start_span("chat probe")
probe.end()
print("span ended:", probe.end_time is not None)  # True: the end time is set now

span ended: True


## Production checklist
- **Sample and keep spans for the right time.** Keep every trace with an error, and sample the rest. Set how long traces live.
- **Content off by default.** Prompt and tool text is opt-in in the GenAI conventions. Turn it on only in a test system.
- **Who can read traces.** A trace store holds customer data. Give it the same access rules as the customer database.
- **Pin the versions.** The attribute names are in the Development status. Test the spans after each upgrade.
- **Alert on the SLO, not on every error.** One failed tool call is a fact to look at; a success rate below target is a page.

## Try it
Write `tool_error_rate(spans)`: the share of `execute_tool` spans with status `ERROR`, as a number from 0 to 1. Then
print it for the spans you loaded. Count spans, not tasks.

In [13]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [14]:
def tool_error_rate(spans: list[dict]) -> float:
    tools = [s for s in spans if s["attributes"].get("gen_ai.operation.name") == "execute_tool"]
    return sum(s["status"]["status_code"] == "ERROR" for s in tools) / len(tools) if tools else 0.0


print("tool error rate:", round(tool_error_rate(spans), 2))

tool error rate: 0.33


In the recorded run, 1 error out of 3 tool spans = 0.33.

## Recap
- A span is one timed step; a trace is the tree of spans for one task (agent, model calls, tool calls).
- Use the OpenTelemetry GenAI names for model and tool steps, and pin the version while they are in Development.
- Log counts, ids, hashes, model names and status. Never the customer's words.
- A dashboard row per task gives success, tool errors, cost from token counts and latency; alert on the SLO.

**Interview:** "How would you find out why one support answer cost $0.02 and took 40 seconds?" Strong answer: open its
trace; the `chat` spans show the tokens and the number of steps, and the `execute_tool` spans show which call was
slow or failed. Then check that the log holds no customer text.